In [1]:
import sys
sys.path.append("../..")

from components.nl_sampler import NL_function_generator
from components.instantanous_effects import InstantanousEffects

import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Example usage of using the instantanous (sample-like) effects component.
# Note, we need an nl_sampler in the case that we want to sample nonlinear relationships

NL = NL_function_generator()
SE = InstantanousEffects(
    n_vars=3,
    link_proba=0.15,
    param_range=[0.3, 0.5],
    mirror_range=True,
    nonlinear_proba=0.5,
    nonstationary_change=0.1,
    nl_sampler= NL,
    sample_tries= 50, # Links are sampled untily an acyclic graph is found. Upper  boundary before throwing an error.
)

In [8]:
# The main functionality of the Structural Equation is "init_instantanous_influence" which 
# initializes a random graph according to the specified parameters.
# "get_step" can then be used to get the next Time Step of the process given the current values of the variables which have to be provided.

SE.init_instantanous_influence()
time_series = np.random.uniform(0,1, (3,1))
# instantanous effects are applied in place to the time series.
effect = SE.get_instantanous_effect(time_series)
print(time_series)
print(SE.links)
print(effect)


[[0.56996392]
 [0.72604469]
 [0.23753187]]
[[0.         0.49171184 0.39646069]
 [0.         0.         0.        ]
 [0.         0.         0.        ]]
[[1.02114074]
 [0.72604469]
 [0.23753187]]


In [ ]:
# If the equation can be set to be empty be setting link proba to 0 or specifying empty struc.
# we use this parameter to occasionally turn of the scm when generating data

SE = InstantanousEffects(
    link_proba=0.0,
    nonlinear_proba=0

)
SE.init_instantanous_influence()
time_series = np.random.uniform(0,1, (3,1))
print(SE.links)


SE = InstantanousEffects(
    link_proba=0.3,
    nonlinear_proba=0

)
SE.init_instantanous_influence()
print(SE.links)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


In [18]:
# If no nonlinearity is involved, the process can be initialized without the NL_sampler.

SE = InstantanousEffects(
    link_proba=0.3,
    nonlinear_proba=0.0,
)

SE.init_instantanous_influence()
time_series = np.random.uniform(0,1, (3,1))
t = SE.get_instantanous_effect(time_series)
SE.links

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

In [29]:
# Masks can be provided to force specific links and specific nl_relationships.
# We use these e.g. to enforce unfaithful relationships in the graph

NL = NL_function_generator()


SE = InstantanousEffects(
    link_proba=0.0,
    nonlinear_proba=0.0
)

# Bool Mask example:
mask = np.zeros((3,3)).astype(bool)
mask[0,1] = True

for x in range(3):
    SE.init_instantanous_influence(link_mask=mask)
    print(SE.links[0][1])
    

# Float Mask example:
mask = np.zeros((3,3)).astype(float)
mask[0,1] = 0.3

for x in range(3):
    SE.init_instantanous_influence(link_mask=mask)
    print(SE.links[0][1])
    


SE = InstantanousEffects(
    link_proba=0.0,
    nonlinear_proba=0.0,
    nl_sampler=NL
)

# NL Mask example: (Note float masking for nl is not supported as it 
# is ambiguous given the nature of the distributions over NL functions.)
# To enforce the link to exist, the link_mask has to be set to True.
nl_mask = np.zeros((3,3)).astype(bool)
nl_mask[0,1] = True
for x in range(3):
    SE.init_instantanous_influence(nl_mask=nl_mask, link_mask=nl_mask)
    print(SE.links)
    print(SE.nl_naming)


-0.3741596048465162
0.39514098524518676
0.4269436640001182
0.3
0.3
0.3
[[ 0.        -0.3741596  0.       ]
 [ 0.         0.         0.       ]
 [ 0.         0.         0.       ]]
[['Nothing' '1.9631664399122064' 'Nothing']
 ['Nothing' 'Nothing' 'Nothing']
 ['Nothing' 'Nothing' 'Nothing']]
[[0.         0.39514099 0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]]
[['Nothing' '1.9007362269751005' 'Nothing']
 ['Nothing' 'Nothing' 'Nothing']
 ['Nothing' 'Nothing' 'Nothing']]
[[0.         0.42694366 0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]]
[['Nothing' '0.5058302741689066' 'Nothing']
 ['Nothing' 'Nothing' 'Nothing']
 ['Nothing' 'Nothing' 'Nothing']]


In [36]:
# Links can additionally be restricted to a previous link tensor. This is only used if nonstationary change is > 0. 
# We use this feature when generating data that has change points in it.
NL = NL_function_generator()



SE = InstantanousEffects(
    link_proba=0.3,
    nonlinear_proba=0.0,
    nonstationary_change=0.05
)

# Mask example:
previous_links= np.zeros((3,3)).astype(float)
previous_links[0,1] = 0.5


for x in range(3):
    SE.init_instantanous_influence(mask_restriction=previous_links)
    print(SE.links[0,1])

0.49388784397520524
0.4870798024232581
0.48683356008779355
